In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

#mount google drive
from google.colab import drive
drive.mount('/content/drive')

# Path to your file (UPDATE THIS PATH)
file_path = '/content/drive/MyDrive/DATA/temperature.csv'

# Load CSV
data = pd.read_csv(file_path)

# Extract data
fahrenheit = data["Fahrenheit"].values.astype(float)
celsius = data["Celsius"].values.astype(float)

# Normalize the data
fahrenheit = fahrenheit / 100.0
celsius = celsius / 100.0

# Model
model = keras.Sequential([
    keras.layers.Dense(units=1, input_shape=[1])
])

# Compile (keeping Adam as you used)
model.compile(optimizer='adam', loss='mean_squared_error')

# Train the model
model.fit(fahrenheit, celsius, epochs=500, verbose=1)

# Input from user
f_input = float(input("Enter temperature in Fahrenheit: "))

# Normalize input
f_input_norm = f_input / 100.0

# Prediction
prediction = model.predict(np.array([f_input_norm]), verbose=0)

# Denormalize output
print("Predicted Celsius value:", prediction[0][0] * 100)

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

# Path to your file (update this to your actual local path)
file_path = 'temperature.csv'   # e.g. "C:/Users/Mayank/Documents/temperature.csv"

# Load CSV
data = pd.read_csv(file_path)

# Extract data
fahrenheit = data["Fahrenheit"].values.astype(float)
celsius = data["Celsius"].values.astype(float)

# Normalize the data
fahrenheit = fahrenheit / 100.0
celsius = celsius / 100.0

# Model
model = keras.Sequential([
    keras.layers.Dense(units=1, input_shape=[1])
])

# Compile
model.compile(optimizer='adam', loss='mean_squared_error')

# Train the model
model.fit(fahrenheit, celsius, epochs=500, verbose=1)

# Input from user
f_input = float(input("Enter temperature in Fahrenheit: "))

# Normalize input
f_input_norm = f_input / 100.0

# Prediction
prediction = model.predict(np.array([f_input_norm]), verbose=0)

# Denormalize output
print("Predicted Celsius value:", prediction[0][0] * 100)

In [ ]:
"""
Advanced Temperature Conversion Model (Fahrenheit → Celsius)
============================================================
Features:
- Auto-generates data if CSV not found
- Multi-layer neural network with BatchNormalization + Dropout
- Learning rate scheduler + EarlyStopping + ModelCheckpoint
- Evaluation with MAE, MSE, R² metrics
- Comparison: Neural Net vs exact formula
- Saves/loads model for reuse
- Interactive prediction loop
"""

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
import sys

# ── Configuration ────────────────────────────────────────────────────────────
FILE_PATH    = "temperature.csv"
MODEL_PATH   = "temp_model.keras"
BATCH_SIZE   = 32
MAX_EPOCHS   = 1000
PATIENCE     = 30          # EarlyStopping patience
LR_INITIAL   = 1e-3

# ── 1. Data Loading / Generation ─────────────────────────────────────────────
def load_or_generate_data(path: str) -> tuple[np.ndarray, np.ndarray]:
    if os.path.exists(path):
        df = pd.read_csv(path)
        if "Fahrenheit" not in df.columns or "Celsius" not in df.columns:
            raise ValueError("CSV must contain 'Fahrenheit' and 'Celsius' columns.")
        print(f"[INFO] Loaded {len(df)} samples from '{path}'.")
        return df["Fahrenheit"].values.astype(float), df["Celsius"].values.astype(float)

    print(f"[WARN] '{path}' not found — generating synthetic data.")
    f = np.linspace(-100, 500, 5000) + np.random.normal(0, 0.1, 5000)
    c = (f - 32) * 5 / 9
    pd.DataFrame({"Fahrenheit": f, "Celsius": c}).to_csv(path, index=False)
    print(f"[INFO] Saved {len(f)} synthetic samples to '{path}'.")
    return f, c


# ── 2. Preprocessing ──────────────────────────────────────────────────────────
def preprocess(fahrenheit: np.ndarray, celsius: np.ndarray):
    f_mean, f_std = fahrenheit.mean(), fahrenheit.std()
    c_mean, c_std = celsius.mean(), celsius.std()

    f_norm = (fahrenheit - f_mean) / f_std
    c_norm = (celsius   - c_mean) / c_std

    # Train / validation / test split  (70 / 15 / 15)
    n = len(f_norm)
    idx = np.random.permutation(n)
    t1, t2 = int(0.70 * n), int(0.85 * n)
    train_idx, val_idx, test_idx = idx[:t1], idx[t1:t2], idx[t2:]

    stats = {"f_mean": f_mean, "f_std": f_std, "c_mean": c_mean, "c_std": c_std}
    splits = {
        "X_train": f_norm[train_idx], "y_train": c_norm[train_idx],
        "X_val":   f_norm[val_idx],   "y_val":   c_norm[val_idx],
        "X_test":  f_norm[test_idx],  "y_test":  c_norm[test_idx],
        "f_test":  fahrenheit[test_idx],
        "c_test":  celsius[test_idx],
    }
    print(f"[INFO] Split → train: {len(train_idx)}  val: {len(val_idx)}  test: {len(test_idx)}")
    return splits, stats


# ── 3. Model Definition ───────────────────────────────────────────────────────
def build_model() -> keras.Model:
    model = keras.Sequential([
        layers.Input(shape=(1,)),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.1),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.1),
        layers.Dense(32, activation="relu"),
        layers.Dense(1),
    ], name="TempConverter")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR_INITIAL),
        loss="mse",
        metrics=["mae"],
    )
    return model


# ── 4. Callbacks ──────────────────────────────────────────────────────────────
def get_callbacks() -> list:
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=PATIENCE,
            restore_best_weights=True, verbose=1
        ),
        keras.callbacks.ModelCheckpoint(
            MODEL_PATH, monitor="val_loss",
            save_best_only=True, verbose=0
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5,
            patience=10, min_lr=1e-6, verbose=1
        ),
        keras.callbacks.TerminateOnNaN(),
    ]


# ── 5. Evaluation ─────────────────────────────────────────────────────────────
def r2_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot if ss_tot != 0 else 0.0


def evaluate(model: keras.Model, splits: dict, stats: dict) -> None:
    X_test = splits["X_test"].reshape(-1, 1)
    c_test  = splits["c_test"]

    pred_norm = model.predict(X_test, verbose=0).flatten()
    pred_c    = pred_norm * stats["c_std"] + stats["c_mean"]

    # Neural network metrics
    mae = np.mean(np.abs(c_test - pred_c))
    mse = np.mean((c_test - pred_c) ** 2)
    r2  = r2_score(c_test, pred_c)

    # Exact formula for comparison
    exact_c    = (splits["f_test"] - 32) * 5 / 9
    exact_mae  = np.mean(np.abs(c_test - exact_c))

    print("\n" + "=" * 50)
    print(" Model Evaluation on Test Set")
    print("=" * 50)
    print(f"  Neural Network  →  MAE: {mae:.4f}°C  |  MSE: {mse:.6f}  |  R²: {r2:.6f}")
    print(f"  Exact Formula   →  MAE: {exact_mae:.6f}°C  (ground-truth reference)")
    print("=" * 50)


# ── 6. Training Plot ──────────────────────────────────────────────────────────
def plot_history(history: keras.callbacks.History) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("Training History", fontsize=13)

    for ax, metric, title in zip(
        axes,
        [("loss", "val_loss"), ("mae", "val_mae")],
        ["Loss (MSE)", "Mean Absolute Error"],
    ):
        ax.plot(history.history[metric[0]], label="Train")
        if metric[1] in history.history:
            ax.plot(history.history[metric[1]], label="Validation")
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("training_history.png", dpi=120)
    print("[INFO] Saved training plot → training_history.png")
    plt.close()


# ── 7. Prediction Helper ──────────────────────────────────────────────────────
def predict(model: keras.Model, stats: dict, fahrenheit_val: float) -> float:
    f_norm = (fahrenheit_val - stats["f_mean"]) / stats["f_std"]
    pred_norm = model.predict(np.array([[f_norm]]), verbose=0)[0][0]
    return pred_norm * stats["c_std"] + stats["c_mean"]


# ── 8. Main ───────────────────────────────────────────────────────────────────
def main() -> None:
    tf.random.set_seed(42)
    np.random.seed(42)

    fahrenheit, celsius = load_or_generate_data(FILE_PATH)
    splits, stats       = preprocess(fahrenheit, celsius)

    # ── Load or train ──────────────────────────────────────────────────────
    if os.path.exists(MODEL_PATH):
        print(f"[INFO] Loading saved model from '{MODEL_PATH}'.")
        model = keras.models.load_model(MODEL_PATH)
    else:
        model = build_model()
        model.summary()

        X_train = splits["X_train"].reshape(-1, 1)
        y_train = splits["y_train"]
        X_val   = splits["X_val"].reshape(-1, 1)
        y_val   = splits["y_val"]

        print(f"\n[INFO] Training for up to {MAX_EPOCHS} epochs …")
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=MAX_EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=get_callbacks(),
            verbose=1,
        )
        plot_history(history)

    evaluate(model, splits, stats)

    # ── Interactive prediction loop ────────────────────────────────────────
    print("\n[INFO] Enter temperatures to predict (type 'quit' to exit).\n")
    while True:
        raw = input("  Fahrenheit → ? ").strip()
        if raw.lower() in ("quit", "exit", "q"):
            break
        try:
            f_val   = float(raw)
            nn_pred = predict(model, stats, f_val)
            exact   = (f_val - 32) * 5 / 9
            print(f"  Neural net : {nn_pred:+.4f}°C")
            print(f"  Formula    : {exact:+.4f}°C")
            print(f"  Difference : {abs(nn_pred - exact):.6f}°C\n")
        except ValueError:
            print("  [!] Please enter a valid number.\n")

    print("[INFO] Done.")


if __name__ == "__main__":
    main()

[INFO] Loaded 80 samples from 'temperature.csv'.
[INFO] Split → train: 56  val: 12  test: 12
[INFO] Loading saved model from 'temp_model.keras'.

 Model Evaluation on Test Set
  Neural Network  →  MAE: 0.8362°C  |  MSE: 1.002768  |  R²: 0.994640
  Exact Formula   →  MAE: 0.000002°C  (ground-truth reference)

[INFO] Enter temperatures to predict (type 'quit' to exit).

  Neural net : +35.5507°C
  Formula    : +37.7778°C
  Difference : 2.227073°C

  Neural net : +49.9445°C
  Formula    : +48.8889°C
  Difference : 1.055573°C

  Neural net : +51.3996°C
  Formula    : +50.0000°C
  Difference : 1.399610°C

  Neural net : +52.1272°C
  Formula    : +50.5556°C
  Difference : 1.571615°C

  Neural net : +43.4428°C
  Formula    : +43.8889°C
  Difference : 0.446129°C

  Neural net : +44.1602°C
  Formula    : +44.4444°C
  Difference : 0.284221°C

  Neural net : +44.8777°C
  Formula    : +45.0000°C
  Difference : 0.122306°C

  Neural net : +45.5952°C
  Formula    : +45.5556°C
  Difference : 0.039596°